Import the coherence, CoLA and LLM-rater and compare them on imported prompt-based text and paraphrases.

# Loading text data

In [6]:
from pathlib import Path
import json
import torch

def load_json_lists_under(folder: Path) -> dict:
    """
    Recursively load all *.json files under `folder`.
    Returns a dict: { "relative/path/stem": [list_of_strings], ... }
    """
    data = {}
    for p in folder.rglob("*.json"):
        try:
            with open(p, "r", encoding="utf-8") as f:
                contents = json.load(f)
            if isinstance(contents, list) and all(isinstance(x, str) for x in contents):
                key = str(p.relative_to(folder).with_suffix(""))  # relative path without .json
                data[key] = contents
        except Exception as e:
            print(f"Skipping {p}: {e}")
    return data


root = Path("../model-comparison/data/")

for folder in root.iterdir():
    if folder.is_dir():
        var_name = f"{folder.name}"
        globals()[var_name] = load_json_lists_under(folder)
        print(f"Loaded {len(globals()[var_name])} JSON files into `{var_name}`")


Loaded 9 JSON files into `shuffled_words`
Loaded 9 JSON files into `shuffled_sentences`
Loaded 9 JSON files into `shuffled_tokens`
Loaded 9 JSON files into `clean`


In [71]:
key_map = {

    "viktor": {
        "clean": "viktor_100_narratives",
        "tokens": "viktor_shuffled_tokens",
        "words": "viktor_shuffled_words",
        "sentences": "viktor_shuffled_sentences"
    },

    "prague": {
        "clean": "prague_100_narratives",
        "tokens": "prague_shuffled_tokens",
        "words": "prague_shuffled_words",
        "sentences": "prague_shuffled_sentences"
    },

    "sciencefic": {
        "clean": "sciencefic_100_narratives",
        "tokens": "sciencefic_shuffled_tokens",
        "words": "sciencefic_shuffled_words",
        "sentences": "sciencefic_shuffled_sentences"
    },

    "gpt4_para1": {
        "clean": "gpt4_para1",
        "tokens": "gpt4_para1_tokens_shuffled",
        "words": "gpt4_para1_words_shuffled",
        "sentences": "gpt4_para1_sentences_shuffled"
    },

    "gpt4_para2": {
        "clean": "gpt4_para2",
        "tokens": "gpt4_para2_tokens_shuffled",
        "words": "gpt4_para2_words_shuffled",
        "sentences": "gpt4_para2_sentences_shuffled"
    },

    "gpt4_para3": {
        "clean": "gpt4_para3",
        "tokens": "gpt4_para3_tokens_shuffled",
        "words": "gpt4_para3_words_shuffled",
        "sentences": "gpt4_para3_sentences_shuffled"
    },

    "gpt5_para1": {
        "clean": "gpt5_para1",
        "tokens": "gpt5_para1_tokens_shuffled",
        "words": "gpt5_para1_words_shuffled",
        "sentences": "gpt5_para1_sentences_shuffled"
    },

    "gpt5_para2": {
        "clean": "gpt5_para2",
        "tokens": "gpt5_para2_tokens_shuffled",
        "words": "gpt5_para2_words_shuffled",
        "sentences": "gpt5_para2_sentences_shuffled"
    },

    "gpt5_para3": {
        "clean": "gpt5_para3",
        "tokens": "gpt5_para3_tokens_shuffled",
        "words": "gpt5_para3_words_shuffled",
        "sentences": "gpt5_para3_sentences_shuffled"
    }
}

# Obtaining values from other models

## The Coherence model - sgnlp_coherence 

In [2]:
from sgnlp.models.coherence_momentum import CoherenceMomentumModel, CoherenceMomentumConfig, \
    CoherenceMomentumPreprocessor

# Load Model
config = CoherenceMomentumConfig.from_pretrained(
    "https://storage.googleapis.com/sgnlp-models/models/coherence_momentum/config.json"
)
model = CoherenceMomentumModel.from_pretrained(
    "https://storage.googleapis.com/sgnlp-models/models/coherence_momentum/pytorch_model.bin",
    config=config
)

preprocessor = CoherenceMomentumPreprocessor(config.model_size, config.max_len)

/home/jdias/miniconda3/envs/sgnlp_coherence/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jdias/miniconda3/envs/sgnlp_coherence/lib/python3.11/site-packages/transformers/utils/hub.py:580: FutureWarning: Using `from_pretrained` with the url of a file (here https://storage.googleapis.com/sgnlp-models/models/coherence_momentum/config.json) is deprecated and won't be possible anymore in v5 of Transformers. You should host your file on the Hub (hf.co) instead and use the repository ID. Note that this is not compatible with the caching system (your file will be downloaded at each execution) or multiple processes (each process will download the file in a different temporary file).
  warnings.warn(
/home/jdias/miniconda3/envs/sgnlp_coherence/lib/python3.11/site-packages/transformers/utils/hub.py:580: Fu

In [ ]:
import torch
model.eval()

def score_texts_batched(texts, batch_size=32):
    """Return list[float] of scores for a list of strings using batched inference."""
    scores = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start+batch_size]
            tensors = preprocessor(batch)
            # tensors["tokenized_texts"] should be [B, ...]
            batch_scores = model.get_main_score(tensors["tokenized_texts"])
            # Convert to Python floats
            scores.extend(batch_scores.detach().cpu().tolist())
    return scores

BATCH_SIZE = 128

results = {}

for clean_key, related_keys in key_map.items():
    # Fetch aligned lists (each len == 100 per your setup)
    clean_texts     = clean[clean_key]
    token_texts     = shuffled_tokens[related_keys["tokens"]]
    word_texts      = shuffled_words[related_keys["words"]]
    sentence_texts  = shuffled_sentences[related_keys["sentences"]]

    n = len(clean_texts)
    assert len(token_texts) == n and len(word_texts) == n and len(sentence_texts) == n, \
        f"Length mismatch for key {clean_key}"

    # Concatenate all four streams so we only call preprocessor/model once per loop
    all_texts = clean_texts + token_texts + word_texts + sentence_texts #order is kept

    # Batched scoring (vectorized across the 4*n texts)
    all_scores = score_texts_batched(all_texts, batch_size=BATCH_SIZE)

    # Split them back into their groups
    clean_scores     = all_scores[0*n : 1*n]
    token_scores     = all_scores[1*n : 2*n]
    word_scores      = all_scores[2*n : 3*n]
    sentence_scores  = all_scores[3*n : 4*n]

    # Store
    results[clean_key] = {
        "clean": clean_scores,
        "tokens": token_scores,
        "words": word_scores,
        "sentences": sentence_scores
    }

    #progress
    print(f"{clean_key}: done. Examples:",
          clean_scores[0], token_scores[0], word_scores[0], sentence_scores[0])


viktor: done. Examples: 11.936232566833496 -26.200389862060547 -26.422626495361328 -1.6471409797668457
prague: done. Examples: 19.50853729248047 -28.530275344848633 -28.441946029663086 -16.485057830810547
sciencefic: done. Examples: 10.367465019226074 -29.009567260742188 -28.1842041015625 -9.786478996276855
gpt4_para1: done. Examples: 17.99173927307129 -26.217805862426758 -26.39042854309082 -18.92676544189453
gpt4_para2: done. Examples: 16.032718658447266 -20.609649658203125 -17.36447525024414 -13.433463096618652
gpt4_para3: done. Examples: 18.807987213134766 -26.24340057373047 -25.26621437072754 -9.653212547302246
gpt5_para1: done. Examples: 18.455106735229492 -27.573211669921875 -27.65407371520996 -10.202364921569824
gpt5_para2: done. Examples: 14.837810516357422 -16.24241065979004 -9.653375625610352 -12.051663398742676
gpt5_para3: done. Examples: 12.992559432983398 -25.762344360351562 -25.844669342041016 -17.742631912231445


Save for later

In [45]:
import pickle

with open("../model-comparison/results/the_coherence.pkl", "wb") as f:
    pickle.dump(results, f)

## CoLA model - change kernel

In [29]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("cointegrated/roberta-large-cola-krishna2020")
model = AutoModelForSequenceClassification.from_pretrained("cointegrated/roberta-large-cola-krishna2020")

def score(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
    return probs.tolist()[0]

print("Grammatical:", score("This is a perfectly normal sentence."))
print("Ungrammatical:", score("This a sentence not good is."))


Grammatical: [0.9904881715774536, 0.009511778131127357]
Ungrammatical: [0.010091143660247326, 0.9899088144302368]


So the Acceptable index is 0 and in index 1 is the prob of being ungrammatical.

In [31]:
# --- config ---
MODEL_NAME = "cointegrated/roberta-large-cola-krishna2020"
BATCH_SIZE = 32
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ACCEPT_IDX = 0  

# --- load model ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

@torch.no_grad()
def score_batch(texts):
    """Return list[float] of 'acceptable' probabilities for a list of strings."""
    out = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start+BATCH_SIZE]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        out.extend(probs[:, ACCEPT_IDX].detach().cpu().tolist())
    return out


# --- run over dicts ---
hf_results = {}

for clean_key, related in key_map.items():
    clean_texts    = clean[clean_key]
    token_texts    = shuffled_tokens[related["tokens"]]
    word_texts     = shuffled_words[related["words"]]
    sentence_texts = shuffled_sentences[related["sentences"]]

    n = len(clean_texts)
    all_texts = clean_texts + token_texts + word_texts + sentence_texts
    scores = score_batch(all_texts)

    hf_results[clean_key] = {
        "clean":     scores[0*n:1*n],
        "tokens":    scores[1*n:2*n],
        "words":     scores[2*n:3*n],
        "sentences": scores[3*n:4*n],
    }

    print(f"{clean_key}: done. First scores ->",
          hf_results[clean_key]['clean'][0],
          hf_results[clean_key]['tokens'][0],
          hf_results[clean_key]['words'][0],
          hf_results[clean_key]['sentences'][0])


viktor: done. First scores -> 0.990220308303833 0.020245017483830452 0.019679434597492218 0.9564821720123291
prague: done. First scores -> 0.9708409905433655 0.02416745387017727 0.02935520000755787 0.9441577196121216
sciencefic: done. First scores -> 0.9546310305595398 0.02358989231288433 0.01762944646179676 0.9509793519973755
gpt4_para1: done. First scores -> 0.9894797205924988 0.01756417378783226 0.018481776118278503 0.9808109402656555
gpt4_para2: done. First scores -> 0.9319757223129272 0.023325953632593155 0.019053732976317406 0.9011991024017334
gpt4_para3: done. First scores -> 0.9867134094238281 0.018329765647649765 0.027319613844156265 0.8848375082015991
gpt5_para1: done. First scores -> 0.9953719973564148 0.016621600836515427 0.016936715692281723 0.986598014831543
gpt5_para2: done. First scores -> 0.9855703115463257 0.018841387704014778 0.016112834215164185 0.8916875123977661
gpt5_para3: done. First scores -> 0.9899945855140686 0.029910365119576454 0.04170824959874153 0.8881634

In [33]:
import pickle 

with open("../model-comparison/results/cola.pkl", "wb") as f:
    pickle.dump(hf_results, f)

## Llama model ratings

In [14]:
import re
import time
import pickle
import torch
import transformers

# ---- configuration ----
MODEL_ID   = "meta-llama/Meta-Llama-3.1-8B-Instruct"
BATCH_SIZE = 16
MAX_NEW_TOKENS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_PKL = "../model-comparison/results/llama.pkl"

print("Running on device:", DEVICE)

Running on device: cuda


In [15]:
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None

NameError: name 'model' is not defined

In [16]:
def create_prompt(text: str):
    """Create a single chat-style prompt for coherence scoring."""
    return [{
        "role": "user",
        "content": (
            "Below is a text extract. Your task is to analyze the extract and assign a coherence score between 0 and 5 inclusive, where:\n\n"
            "0: The text is completely incoherent and lacks any logical connection.\n"
            "1: The text has some minor connections, but overall it is disjointed and hard to follow.\n"
            "2: The text has some coherence, but it is still difficult to understand due to unclear relationships between ideas.\n"
            "3: The text is moderately coherent, with some clear connections between ideas, but may lack depth or clarity.\n"
            "4: The text is highly coherent, with clear and logical connections between ideas, making it easy to follow.\n"
            "5: The text is extremely coherent, with a clear and concise structure, making it effortless to understand.\n\n"
            "You will provide a score ONLY. Do NOT also provide an explanation.\n"
            f"The extract: {text}\n"
            "After examining the extract, the coherence score between 0 and 5 inclusive is:"
        )
    }]

def create_prompt_batch(batch_texts):
    """Wraps multiple texts into a list of chat prompts."""
    return [create_prompt(t) for t in batch_texts]


In [17]:
# Load tokenizer and model once
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
tokenizer.pad_token_id = tokenizer.eos_token_id  # required for Llama

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("Model and tokenizer loaded.")


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.62it/s]


Model and tokenizer loaded.


In [18]:
_score_regex = re.compile(r"\b([0-5])\b")

def extract_score(generated_text: str, prompt_prefix: str) -> int:
    """
    Extract a single integer 0..5 from model output.
    Removes prompt prefix and finds the first standalone digit 0–5.
    """
    continuation = generated_text[len(prompt_prefix):].strip()
    m = _score_regex.search(continuation)
    if m:
        return int(m.group(1))
    digits = re.findall(r"[0-5]", continuation)
    return int(digits[-1]) if digits else 0


In [19]:
@torch.inference_mode()
def call_llama_on_texts(texts):
    """Run coherence scoring on a list of strings."""
    scores = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        msgs = create_prompt_batch(batch)

        rendered = tokenizer.apply_chat_template(
            msgs, add_generation_prompt=True, tokenize=False
        )

        enc = tokenizer(rendered, padding="longest", return_tensors="pt")
        enc = {k: v.to(model.device) for k, v in enc.items()}

        prefix_texts = tokenizer.batch_decode(enc["input_ids"], skip_special_tokens=True)

        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=terminators,
            do_sample=False
        )

        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        batch_scores = [extract_score(decoded[j], prefix_texts[j]) for j in range(len(batch))]
        scores.extend(batch_scores)

        print(f"Rated {min(i+BATCH_SIZE, len(texts))}/{len(texts)} | scores: {batch_scores}")
    return scores


In [20]:
# Cell 6 — Loop using your exact structure and save as .pkl
def run_all_and_save():
    """
    Uses:
      - clean, shuffled_tokens, shuffled_words, shuffled_sentences
      - key_map (maps each clean_key to related_keys for tokens/words/sentences)
    Saves:
      - results_llama.pkl with structure results[clean_key][variant] = list[int]
    """
    t0 = time.time()
    results = {}

    for clean_key, related_keys in key_map.items():
        print(f"--- {clean_key} ---")

        # Your exact structure:
        clean_texts     = clean[clean_key]
        token_texts     = shuffled_tokens[related_keys["tokens"]]
        word_texts      = shuffled_words[related_keys["words"]]
        sentence_texts  = shuffled_sentences[related_keys["sentences"]]

        # Score each variant independently
        scores_clean    = call_llama_on_texts(clean_texts)
        scores_tokens   = call_llama_on_texts(token_texts)
        scores_words    = call_llama_on_texts(word_texts)
        scores_sentences= call_llama_on_texts(sentence_texts)

        # Store
        results[clean_key] = {
            "clean":     scores_clean,
            "tokens":    scores_tokens,
            "words":     scores_words,
            "sentences": scores_sentences
        }

        n = len(clean_texts)
        assert all(len(v) == n for v in results[clean_key].values()), f"Length mismatch at {clean_key}"
        print(f"{clean_key}: done ({n} per variant)")

    with open(OUTPUT_PKL, "wb") as f:
        pickle.dump(results, f)

    print(f"Saved to {OUTPUT_PKL} in {time.time() - t0:.2f}s")


In [13]:
# Execute the run
run_all_and_save()

--- viktor ---
Rated 16/100 | scores: [4, 5, 4, 4, 4, 5, 4, 5, 4, 4, 5, 4, 5, 4, 5, 4]
Rated 32/100 | scores: [4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 5, 5, 4]
Rated 48/100 | scores: [4, 4, 5, 4, 4, 5, 5, 4, 4, 5, 4, 5, 4, 4, 4, 4]
Rated 64/100 | scores: [4, 5, 4, 4, 5, 4, 4, 4, 4, 4, 4, 5, 5, 4, 4, 4]
Rated 80/100 | scores: [5, 4, 5, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4]
Rated 96/100 | scores: [4, 4, 5, 5, 4, 4, 4, 4, 4, 5, 4, 4, 5, 5, 4, 5]
Rated 100/100 | scores: [4, 4, 4, 5]
Rated 16/100 | scores: [3, 3, 2, 3, 3, 2, 3, 2, 3, 3, 2, 3, 3, 2, 2, 3]
Rated 32/100 | scores: [3, 3, 3, 3, 2, 3, 2, 3, 3, 3, 3, 3, 2, 3, 3, 3]
Rated 48/100 | scores: [2, 3, 2, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2]
Rated 64/100 | scores: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3]
Rated 80/100 | scores: [3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 2, 2, 3, 2, 3]
Rated 96/100 | scores: [3, 3, 3, 3, 2, 3, 2, 2, 2, 2, 3, 2, 2, 3, 3, 3]
Rated 100/100 | scores: [3, 3, 3, 3]
Rated 16/100 | scores: [3, 3, 3, 3, 3, 3, 3, 4,

## GPT model ratings

In [38]:
from openai import OpenAI

def get_coherence_score(extract):
    client = OpenAI()
    
    prompt = ("Below is a text extract. Your task is to analyze the extract and assign a coherence score between 0 and 5 inclusive, where:\n\n"
              "0: The text is completely incoherent and lacks any logical connection.\n"
              "1: The text has some minor connections, but overall it is disjointed and hard to follow.\n"
              "2: The text has some coherence, but it is still difficult to understand due to unclear relationships between ideas.\n"
              "3: The text is moderately coherent, with some clear connections between ideas, but may lack depth or clarity.\n"
              "4: The text is highly coherent, with clear and logical connections between ideas, making it easy to follow.\n"
              "5: The text is extremely coherent, with a clear and concise structure, making it effortless to understand.\n\n"
              "You will provide a score ONLY. Do NOT also provide an explanation.\n"
              f"The extract: {extract}\n"
              "After examining the extract, the coherence score between 0 and 5 inclusive is:")
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",  #using gpt-4o-mini to optimize cost
        messages=[
            {"role": "system", "content": "Analyze the coherence of the given text."},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.choices[0].message.content.strip()

In [ ]:
import asyncio
from openai import AsyncOpenAI
import itertools

client = OpenAI()

MODEL = "gpt-4o-mini"   # or a newer small model you prefer
SYSTEM_MSG = "Analyze the coherence of the given text."
USER_TEMPLATE = (
    "Below is a text extract. Your task is to analyze the extract and assign a coherence score between 0 and 5 inclusive, where:\n\n"
    "0: The text is completely incoherent and lacks any logical connection.\n"
    "1: The text has some minor connections, but overall it is disjointed and hard to follow.\n"
    "2: The text has some coherence, but it is still difficult to understand due to unclear relationships between ideas.\n"
    "3: The text is moderately coherent, with some clear connections between ideas, but may lack depth or clarity.\n"
    "4: The text is highly coherent, with clear and logical connections between ideas, making it easy to follow.\n"
    "5: The text is extremely coherent, with a clear and concise structure, making it effortless to understand.\n\n"
    "You will provide a score ONLY. Do NOT also provide an explanation.\n"
    "The extract: {extract}\n"
    "After examining the extract, the coherence score between 0 and 5 inclusive is:"
)


async_client = AsyncOpenAI()
CONCURRENCY = 3  
# _req_counter = itertools.count(1)

def _coerce_score(text: str) -> float:
    """
    Try to extract a single number 0..5 from the model output.
    Accepts integers or floats, clamps to [0,5].
    """
    s = text.strip()
    # common cases: "4", "4.0", "Score: 4"
    num = None
    # find first number-like token
    import re
    m = re.search(r"[-+]?\d*\.?\d+", s)
    if m:
        try:
            num = float(m.group(0))
        except ValueError:
            pass
    if num is None:
        # fallback: treat any non-numeric as 0
        num = 0.0
    # clamp
    return max(0.0, min(5.0, num))

async def _async_score_one(extract: str, sem: asyncio.Semaphore) -> float:
    async with sem:
        resp = await async_client.chat.completions.create(
            model=MODEL,
            #temperature=0,
            #max_tokens=5,
            messages=[
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": USER_TEMPLATE.format(extract=extract)}
            ]
        )
    out = resp.choices[0].message.content or ""
    #return _coerce_score(out)
    return int(out)

# async def _async_score_one(extract: str, sem: asyncio.Semaphore) -> float:
#     idx = next(_req_counter)  # local, monotonic counter
#     async with sem:
#         try:
#             resp = await async_client.chat.completions.create(
#                 model=MODEL,
#                 temperature=0,
#                 max_tokens=5,
#                 messages=[
#                     {"role": "system", "content": SYSTEM_MSG},
#                     {"role": "user", "content": USER_TEMPLATE.format(extract=extract)}
#                 ]
#             )
#         except Exception as e:
#             print(f"[REQ {idx}] ERROR: {e!r}")
#             raise
#     # The SDK exposes response.id; the HTTP request-id header is also exposed internally.
#     print(f"[REQ {idx}] response.id={resp.id}")
#     out = resp.choices[0].message.content or ""
#     return int(out)

async def score_texts_batched_async(texts: list[str]) -> list[float]:
    sem = asyncio.Semaphore(CONCURRENCY)
    tasks = [asyncio.create_task(_async_score_one(t, sem)) for t in texts]
    return await asyncio.gather(*tasks)

def score_texts_batched(texts: list[str]) -> list[float]:
    """
    Notebook-safe wrapper: if a loop is running (e.g., Jupyter),
    patch it with nest_asyncio and run the coroutine; otherwise use asyncio.run().
    """
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        import nest_asyncio  # pip install nest_asyncio
        nest_asyncio.apply()
        return loop.run_until_complete(score_texts_batched_async(texts))
    else:
        return asyncio.run(score_texts_batched_async(texts))

In [75]:
results = {}

for clean_key, related_keys in key_map.items():
    clean_texts     = clean[clean_key]
    token_texts     = shuffled_tokens[related_keys["tokens"]]
    word_texts      = shuffled_words[related_keys["words"]]
    sentence_texts  = shuffled_sentences[related_keys["sentences"]]

    n = len(clean_texts)
    assert len(token_texts) == n and len(word_texts) == n and len(sentence_texts) == n, \
        f"Length mismatch for key {clean_key}"

    all_texts = clean_texts + token_texts + word_texts + sentence_texts  # order kept

    all_scores = score_texts_batched(all_texts)

    clean_scores     = all_scores[0*n : 1*n]
    token_scores     = all_scores[1*n : 2*n]
    word_scores      = all_scores[2*n : 3*n]
    sentence_scores  = all_scores[3*n : 4*n]

    results[clean_key] = {
        "clean": clean_scores,
        "tokens": token_scores,
        "words": word_scores,
        "sentences": sentence_scores
    }

    print(f"{clean_key}: done. Examples:",
          clean_scores[0], token_scores[0], word_scores[0], sentence_scores[0])


viktor: done. Examples: 5 0 0 4
prague: done. Examples: 4 0 1 4
sciencefic: done. Examples: 5 0 1 4
gpt4_para1: done. Examples: 5 1 1 3
gpt4_para2: done. Examples: 4 0 1 3
gpt4_para3: done. Examples: 5 0 0 3
gpt5_para1: done. Examples: 5 0 0 4
gpt5_para2: done. Examples: 4 0 1 4
gpt5_para3: done. Examples: 5 0 0 3


In [76]:
with open("../model-comparison/results/gpt.pkl", "wb") as f:
    pickle.dump(results, f)